# 实验7.1 嵌入式轻量化大模型训练实验

> **GitCode 昇腾 910B3 云沙箱 · Qwen1.5-0.5B-Chat · LoRA 参数高效微调**

本实验在 **GitCode 昇腾 910B3 云沙箱**环境中，使用 **LoRA（Low-Rank Adaptation，低秩适配）** 方法对 **Qwen1.5-0.5B-Chat** 大语言模型进行参数高效微调，使其成为"华为昇腾 AI 助手"。实验涵盖数据集构建、LoRA 配置、模型训练、权重保存与微调前后效果对比的完整流程。

**运行环境**：`cann_9.0.0-py3.11-A2-arm` · `ASCEND 1*NPU 910B3` · `16vCPUs, 32GiB`

---

## 1. 实验概述

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">项目</th>
<th style="text-align: left;">内容</th>
</tr>
<tr>
<td style="text-align: left;"><strong>实验名称</strong></td>
<td style="text-align: left;">嵌入式轻量化大模型训练实验</td>
</tr>
<tr>
<td style="text-align: left;"><strong>目标硬件</strong></td>
<td style="text-align: left;">昇腾 910B3 NPU（云沙箱）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>软件环境</strong></td>
<td style="text-align: left;">CANN · PyTorch · torch_npu · PEFT · Transformers</td>
</tr>
<tr>
<td style="text-align: left;"><strong>基础模型</strong></td>
<td style="text-align: left;">Qwen1.5-0.5B-Chat（462M 参数）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>微调方法</strong></td>
<td style="text-align: left;">LoRA（r=8, alpha=16, target: q_proj/v_proj）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>可训练参数</strong></td>
<td style="text-align: left;">0.3M（占总参数 0.065%）</td>
</tr>
</table>

**表格解读**：本实验在昇腾910B3 NPU云沙箱上，使用LoRA方法微调Qwen1.5-0.5B-Chat模型。LoRA配置为秩r=8、缩放因子alpha=16，仅在注意力模块的q_proj和v_proj上注入旁路矩阵。这样可训练参数仅0.3M，占模型总参数（462M）的0.065%，极大降低了训练资源需求。基础模型以float16加载，平衡精度和显存占用。

### 实验目标

- **知识目标**：理解 LoRA 低秩适配的数学原理与参数高效微调思想；掌握大模型训练在昇腾 NPU 上的设备适配方法。
- **能力目标**：能够在云沙箱中使用 LoRA 微调 Qwen1.5-0.5B 模型；能够设计训练前后对比实验验证微调效果。
- **素养目标**：形成"实践→总结→贡献"的闭环意识，积极向 CANN 社区反馈大模型训练经验。

## 2. LoRA 原理详解

### 2.1 为什么需要参数高效微调？

全参数微调需要更新模型所有参数，对于大模型而言：
- **显存占用大**：需要存储所有参数的梯度 + 优化器状态
- **训练速度慢**：参数量巨大，每步计算量大
- **存储成本高**：每个任务一份完整模型

### 2.2 LoRA 的核心思想

**LoRA（Low-Rank Adaptation）** 冻结原始参数，仅在旁路注入低秩矩阵进行训练：

$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} \cdot B \cdot A$$

其中：
- $W_0$ 是冻结的基座权重（$d \times k$），不参与训练
- $B$ 是训练得到的矩阵（$d \times r$）
- $A$ 是训练得到的矩阵（$r \times k$）
- $r$ 是低秩矩阵的秩（$r \ll \min(d, k)$）
- $\alpha/r$ 是缩放系数

### 2.3 LoRA 的优势

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">显存占用减少 80%</td>
<td style="text-align: left;">只训练 0.1%~1% 的参数</td>
</tr>
<tr>
<td style="text-align: left;">训练速度提升 3 倍</td>
<td style="text-align: left;">参数量大幅减少</td>
</tr>
<tr>
<td style="text-align: left;">多任务轻松切换</td>
<td style="text-align: left;">每个任务只存几 MB 的 LoRA 权重</td>
</tr>
<tr>
<td style="text-align: left;">推理零额外延迟</td>
<td style="text-align: left;">合并后 W = W₀ + (α/r)·B·A，与原模型结构一致</td>
</tr>
</table>

**表格解读**：LoRA的四大优势使其成为大模型微调的主流方法。显存占用减少80%是因为只训练0.1%~1%的参数，优化器状态和梯度只针对这些参数，大幅减少显存。训练速度提升3倍源于可训练参数量大幅减少，每步反向传播只需更新低秩矩阵B和A。多任务切换时每个任务只需存储几MB的LoRA权重（vs 完整模型924MB），可像插件一样动态加载。推理零延迟是通过`merge_and_unload()`将LoRA增量合并回基座权重，合并后模型结构与原模型完全一致，推理时无额外计算开销。

## 3. 环境初始化

### 3.1 设置 HuggingFace 国内镜像

防止模型下载超时，设置 `HF_ENDPOINT` 为国内镜像。

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
try:
    import torch_npu
    NPU_AVAILABLE = torch.npu.is_available()
except ImportError:
    NPU_AVAILABLE = False
    print('[警告] torch_npu 未安装，将使用 CPU 训练（速度较慢）')

if NPU_AVAILABLE:
    device = torch.device('npu:0')
    torch.npu.set_device(0)
    print(f'设备: 昇腾 NPU ({device})')
    try:
        import subprocess
        result = subprocess.run(['npu-smi', 'info'], capture_output=True, text=True, timeout=5)
        print(result.stdout[:300])
    except Exception:
        pass
else:
    device = torch.device('cpu')
    print(f'设备: CPU (NPU 不可用)')

print(f'PyTorch 版本: {torch.__version__}')

**代码说明与预期结果**：

- **环境检测**：设置HuggingFace国内镜像`hf-mirror.com`加速模型下载。检测`torch_npu`是否安装且NPU可用，可用则设为`npu:0`并打印`npu-smi info`信息，否则回退CPU。
- **预期输出**：若NPU可用，打印设备名和NPU状态信息（芯片型号、显存等）；PyTorch版本号。若NPU不可用，打印警告并使用CPU（训练速度会显著变慢）。

### 3.2 安装依赖库

> LoRA微调需要 `transformers` 和 `peft` 库。若未安装，下方代码会自动安装。

In [ ]:
import subprocess
import sys

def ensure_package(package, import_name=None):
    import_name = import_name or package
    try:
        __import__(import_name)
        print(f'[OK] {package} 已安装')
    except ImportError:
        print(f'[安装] {package} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f'[完成] {package} 安装成功')

# 修复: 部分昇腾环境中 torchaudio 的 C 扩展与当前 torch 版本不匹配，导入时会报
# OSError: undefined symbol: torch_library_impl，连锁导致 transformers 导入失败。
# NLP 任务不需要 torchaudio，先尝试卸载；若卸载失败则在 sys.modules 中屏蔽。
try:
    import importlib
    importlib.import_module('torchaudio')
except Exception:
    for _m in [m for m in list(sys.modules) if m == 'torchaudio' or m.startswith('torchaudio.')]:
        sys.modules.pop(_m, None)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        pass
    try:
        importlib.import_module('torchaudio')
    except Exception:
        sys.modules['torchaudio'] = None
        print('[修复] 已屏蔽不兼容的 torchaudio（NLP 任务不需要）')
    else:
        print('[修复] 已卸载不兼容的 torchaudio（NLP 任务不需要）')

ensure_package('transformers')
ensure_package('peft')
print('依赖检查完成!')

In [ ]:
!pip install "transformers==4.39.3" -q
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
import json
import time

print('依赖库加载完成!')

**代码说明**：导入LoRA微调所需的核心库：`transformers`提供模型加载（AutoModelForCausalLM/AutoTokenizer）和训练工具（Trainer/TrainingArguments），`peft`提供LoRA配置（LoraConfig）和模型包装（get_peft_model），`torch.utils.data.Dataset`用于自定义数据集，`json`和`time`用于数据读写和计时。

> **注意**：若报 `ModuleNotFoundError`，请先运行上方「3.2 安装依赖库」单元格安装 `transformers` 和 `peft`，然后重新运行本单元格。

## 4. 数据集准备

我们使用内置的 **41 条昇腾 AI 问答对** 作为训练数据，让模型学会回答华为昇腾 AI 生态相关问题。

In [ ]:
SYSTEM_PROMPT = '你是华为昇腾AI助手，请简洁准确地回答问题。'

TRAIN_DATA = [
    {'question': '什么是昇腾910？', 'answer': '昇腾910是华为推出的高性能AI训练处理器，采用达芬奇架构，半精度(FP16)算力可达256TFLOPS，主要用于深度学习训练。'},
    {'question': '什么是昇腾310？', 'answer': '昇腾310是华为推出的高效能AI推理处理器，整型算力(INT8)达16TOPS，功耗仅8W，适用于边缘计算和推理场景。'},
    {'question': '什么是MindSpore？', 'answer': 'MindSpore是华为自主研发的全场景深度学习框架，支持云、边缘和端侧部署，具有易开发、高效执行的特点。'},
    {'question': '什么是CANN？', 'answer': 'CANN是华为异构计算架构，为昇腾AI处理器提供计算使能，包含算子库、模型转换和推理引擎等工具。'},
    {'question': '什么是香橙派？', 'answer': '香橙派OrangePi AIpro是基于昇腾AI技术的开发板，支持8-12TOPS AI算力，预置MindSpore框架，适合AI开发和学习。'},
    {'question': '昇腾910的算力是多少？', 'answer': '昇腾910半精度(FP16)算力可达256TFLOPS，整数算力(INT8)可达512TOPS，是华为最强AI训练芯片。'},
    {'question': 'MindSpore有什么特点？', 'answer': 'MindSpore具有易开发、高效执行、全场景统一部署三大特点，支持自动微分、自动并行和动静态图统一。'},
    {'question': 'CANN的作用是什么？', 'answer': 'CANN为昇腾AI处理器提供底层计算使能，包括算子开发、模型转换、推理引擎和性能优化等功能。'},
    {'question': '什么是达芬奇架构？', 'answer': '达芬奇架构是华为自研的AI计算架构，采用3D Cube计算引擎，高效支持矩阵运算，是昇腾处理器的核心。'},
    {'question': '昇腾910和310有什么区别？', 'answer': '昇腾910用于AI训练，算力256TFLOPS；昇腾310用于AI推理，算力16TOPS功耗8W。910性能更强，310更节能。'},
    {'question': '什么是模型微调？', 'answer': '模型微调是在预训练模型基础上用特定领域数据继续训练，使模型适配特定任务，比从头训练更高效。'},
    {'question': '什么是LoRA？', 'answer': 'LoRA是低秩适配方法，冻结原始参数，仅训练旁路注入的低秩矩阵，可训练参数量仅为原模型0.1%~1%。'},
    {'question': 'MindSpore支持哪些硬件？', 'answer': 'MindSpore支持华为昇腾910/310/910B等AI处理器，也支持CPU和GPU，实现全场景统一部署。'},
    {'question': '什么是AI训练？', 'answer': 'AI训练是用大量数据迭代优化模型参数的过程，通过反向传播和梯度下降使模型逐步学习数据中的规律。'},
    {'question': '什么是AI推理？', 'answer': 'AI推理是使用训练好的模型对新输入数据进行预测的过程，关注低延迟和高吞吐，通常在端侧或边缘执行。'},
    {'question': '什么是半精度浮点数？', 'answer': '半精度浮点数(FP16)用16位表示一个浮点数，相比FP32节省一半显存，是AI训练中常用的数据类型。'},
    {'question': '什么是张量？', 'answer': '张量是多维数组的统称，是深度学习中最基本的数据结构。标量是0维张量，向量是1维，矩阵是2维。'},
    {'question': '什么是神经网络？', 'answer': '神经网络是模仿人脑神经元结构的计算模型，由多层节点组成，通过权重和激活函数学习数据的特征。'},
    {'question': '什么是深度学习？', 'answer': '深度学习是机器学习的分支，使用多层神经网络自动学习数据特征，在图像、NLP等领域表现优异。'},
    {'question': '什么是自然语言处理？', 'answer': '自然语言处理(NLP)是AI的重要分支，研究计算机理解和生成人类语言，包括文本分类、翻译、对话等任务。'},
    {'question': '什么是大语言模型？', 'answer': '大语言模型(LLM)是参数量巨大的语言模型，通过海量文本预训练，具备强大的语言理解和生成能力。'},
    {'question': 'Qwen是什么模型？', 'answer': 'Qwen通义千问是阿里云研发的大语言模型，Qwen1.5-0.5B是其轻量版，有5亿参数，适合端侧部署。'},
    {'question': '什么是分词器？', 'answer': '分词器Tokenizer将文本切分为子词单元并映射为数字ID，是语言模型处理文本的第一步。'},
    {'question': '什么是注意力机制？', 'answer': '注意力机制让模型关注输入中最重要的部分，通过计算查询-键-值的加权求和实现，是Transformer的核心。'},
    {'question': '什么是Transformer？', 'answer': 'Transformer是基于自注意力机制的神经网络架构，摒弃了循环结构，支持并行计算，是大模型的基础架构。'},
    {'question': '什么是梯度下降？', 'answer': '梯度下降是优化算法，沿损失函数梯度反方向更新参数，逐步最小化损失，是模型训练的核心方法。'},
    {'question': '什么是学习率？', 'answer': '学习率控制每步参数更新的幅度，过大导致震荡发散，过小收敛缓慢，通常取0.001~0.01。'},
    {'question': '什么是过拟合？', 'answer': '过拟合是模型在训练集表现好但泛化能力差，可通过正则化、Dropout、数据增强等方法缓解。'},
    {'question': '什么是损失函数？', 'answer': '损失函数衡量模型预测值与真实值的差距，训练目标是最小化损失，常用有交叉熵、均方误差等。'},
    {'question': '什么是反向传播？', 'answer': '反向传播通过链式法则从输出层向输入层逐层计算梯度，是神经网络训练中计算参数梯度的核心算法。'},
    {'question': '什么是批量大小？', 'answer': '批量大小Batch Size是每次训练更新使用的样本数，影响训练速度和稳定性，常用值为8、16、32。'},
    {'question': '什么是epoch？', 'answer': 'epoch指模型遍历整个训练数据集一次，训练多个epoch使模型充分学习数据特征。'},
    {'question': '什么是预训练？', 'answer': '预训练是用海量无标注数据训练模型学习通用特征，得到预训练模型后再用有标注数据微调。'},
    {'question': '什么是参数高效微调？', 'answer': '参数高效微调(PEFT)只训练少量附加参数实现微调，包括LoRA、Prefix Tuning等方法，大幅降低资源需求。'},
    {'question': 'MindSpore的优势是什么？', 'answer': 'MindSpore优势包括：API友好易开发、自动并行高效执行、全场景统一部署、支持大模型训练和推理。'},
    {'question': '如何安装MindSpore？', 'answer': '可通过pip install mindspore安装，昇腾环境需先安装CANN工具包，具体参考MindSpore官网安装指南。'},
    {'question': '什么是全场景部署？', 'answer': '全场景部署指同一套代码和模型可在云、边、端不同环境运行，MindSpore通过统一API实现这一目标。'},
    {'question': '什么是昇思？', 'answer': '昇思MindSpore是华为开源的深度学习框架，名字取自昇腾和思，寓意昇腾AI的思考能力。'},
    {'question': '什么是MindNLP？', 'answer': 'MindNLP是基于MindSpore的自然语言处理套件，提供与HuggingFace兼容的API，支持加载主流大模型。'},
    {'question': 'LoRA的秩r是什么？', 'answer': 'LoRA的秩r控制低秩矩阵的大小，r越大表达能力越强但参数越多，常用值为4、8、16，教学实验中取8即可。'},
    {'question': '什么是梯度裁剪？', 'answer': '梯度裁剪限制梯度的最大范数，防止梯度爆炸导致训练不稳定，通常设置最大梯度范数为1.0。'},
]

with open('ascend_qa.json', 'w', encoding='utf-8') as f:
    json.dump(TRAIN_DATA, f, ensure_ascii=False, indent=2)

print(f'数据集: {len(TRAIN_DATA)} 条问答对，已保存至 ascend_qa.json')
print(f'前 3 条示例:')
for i in range(3):
    print(f'  Q: {TRAIN_DATA[i]["question"]}')
    print(f'  A: {TRAIN_DATA[i]["answer"][:60]}...')

**代码说明与预期结果**：

- **数据集设计**：内置41条昇腾AI问答对，覆盖昇腾芯片（910/310）、MindSpore、CANN、香橙派、LoRA、深度学习基础概念等。每条包含question和answer字段，system_prompt统一设为"你是华为昇腾AI助手"。
- **数据保存**：以JSON格式保存到`ascend_qa.json`，便于复用和检查。
- **预期输出**：打印"数据集: 41 条问答对"，并显示前3条示例的问答内容。
- **为什么41条**：教学实验中数据量适中即可演示LoRA微调效果。实际产品中通常需要数百至数千条高质量数据。数据质量比数量更重要——答案要准确、简洁、风格一致。

> **注意**：若报 `NameError: name 'json' is not defined`，说明上方依赖库导入单元格未成功运行，请先运行「3.2 安装依赖库」→依赖库导入单元格。

## 5. 加载预训练模型

从 HuggingFace 下载 `Qwen/Qwen1.5-0.5B-Chat`，以 `float16` 加载到昇腾 NPU。

In [ ]:
MODEL_NAME = 'Qwen/Qwen1.5-0.5B-Chat'
print(f'加载模型: {MODEL_NAME} ...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'模型: {MODEL_NAME}')
print(f'参数量: {total_params / 1e6:.1f}M ({total_params / 1e9:.3f}B)')
print(f'数据类型: float16')

**代码说明与预期结果**：加载Qwen1.5-0.5B-Chat基座模型，以float16精度加载到NPU。预期参数量约462M（0.462B），首次运行需从HuggingFace下载约924MB模型权重。

> **注意**：若报 `NameError: name 'AutoTokenizer' is not defined`，请先运行「3.2 安装依赖库」→依赖库导入单元格。

## 6. 微调前测试（基线）

用 3 个测试问题评估原始模型能力，作为对照基线。

In [ ]:
TEST_QUESTIONS = [
    '什么是昇腾910？',
    '什么是LoRA？',
    'MindSpore有什么特点？',
]

def ask_question(question, model, tokenizer, device):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', tokenize=True, return_dict=False,
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids, max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = input_ids.shape[-1]
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return response

print('=== 微调前模型测试 ===')
for q in TEST_QUESTIONS:
    print(f'  问: {q}')
    try:
        answer = ask_question(q, model, tokenizer, device)
        print(f'  答: {answer[:100]}')
    except Exception as e:
        print(f'  答: (出错: {e})')
    print()

**代码说明与预期结果**：

- **基线测试**：用3个训练集中的问题测试未微调的原始模型，记录其回答作为对照基线。`ask_question`函数封装了chat template格式化、模型生成、解码的完整流程。`do_sample=False`贪心解码确保结果可复现。
- **预期输出**：原始模型对昇腾相关问题回答可能不够准确或偏离主题（因为预训练数据中昇腾相关内容有限）。例如"什么是昇腾910？"可能给出笼统的回答而非具体的256TFLOPS算力信息。这正是我们需要微调的原因——让模型学会领域知识。
- **为什么先测基线**：建立before/after对比，量化微调效果。没有基线就无法证明微调确实有效。

## 7. 数据预处理与 LoRA 配置

### 7.1 构造问答数据集

将问答对格式化为模型可训练的格式：使用 chat template 格式化对话，分词后构造 input_ids、attention_mask 和 labels。

In [ ]:
MAX_LENGTH = 256

class QADataset(Dataset):
    def __init__(self, data, tokenizer, max_length=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': item['question']},
            {'role': 'assistant', 'content': item['answer']},
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        enc = self.tokenizer(
            text, truncation=True,
            max_length=self.max_length,
            padding='max_length',
        )
        input_ids = enc['input_ids']
        attention_mask = enc['attention_mask']
        labels = input_ids[:]
        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
        }

train_dataset = QADataset(TRAIN_DATA, tokenizer, max_length=MAX_LENGTH)
print(f'训练样本数: {len(train_dataset)}, 序列长度: {MAX_LENGTH}')
print(f'单样本 input_ids shape: {train_dataset[0]["input_ids"].shape}')

**代码说明与预期结果**：

- **QADataset**：自定义数据集类，继承`torch.utils.data.Dataset`。`__getitem__`将每条问答对通过chat template格式化为完整对话文本（system+user+assistant），分词后截断/填充到固定长度256，构造input_ids、attention_mask和labels。
- **labels = input_ids**：语言模型的训练目标是预测下一个token，因此labels与input_ids相同（Trainer内部会自动右移一位计算loss）。
- **预期输出**：训练样本数41，序列长度256，单样本input_ids shape为`torch.Size([256])`。
- **为什么MAX_LENGTH=256**：昇腾问答的问答对较短，256 tokens足够覆盖system prompt+问题+答案。过长浪费计算，过短会截断答案。

> **注意**：若报 `NameError: name 'Dataset' is not defined`，请先运行「3.2 安装依赖库」→依赖库导入单元格。

### 7.2 配置 LoRA

关键超参数：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">参数</th>
<th style="text-align: left;">值</th>
<th style="text-align: left;">说明</th>
</tr>
<tr>
<td style="text-align: left;">LoRA 秩 r</td>
<td style="text-align: left;">8</td>
<td style="text-align: left;">低秩矩阵维度</td>
</tr>
<tr>
<td style="text-align: left;">lora_alpha</td>
<td style="text-align: left;">16</td>
<td style="text-align: left;">LoRA 缩放因子（α/r = 2）</td>
</tr>
<tr>
<td style="text-align: left;">target_modules</td>
<td style="text-align: left;">q_proj, v_proj</td>
<td style="text-align: left;">注入旁路的注意力模块</td>
</tr>
<tr>
<td style="text-align: left;">lora_dropout</td>
<td style="text-align: left;">0.05</td>
<td style="text-align: left;">LoRA Dropout</td>
</tr>
</table>

**表格解读**：LoRA的四个关键超参数。秩r=8控制低秩矩阵B(d×8)和A(8×k)的中间维度，r越大表达能力越强但参数越多，r=8是教学和轻量微调的常用值。lora_alpha=16是缩放因子，实际缩放为alpha/r=2，控制LoRA增量对原权重的影响程度。target_modules指定在注意力的q_proj（查询投影）和v_proj（值投影）上注入LoRA旁路，这是经验上效果最好的位置。lora_dropout=0.05在LoRA旁路上加Dropout防止过拟合。

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f'\nLoRA 配置:')
print(f'  r = 8, alpha = 16, 缩放 = {16/8}')
print(f'  target_modules = q_proj, v_proj')
print(f'  dropout = 0.05')

**代码说明与预期结果**：

- **LoRA配置**：`LoraConfig`创建LoRA配置对象，`get_peft_model`将基座模型包装为PEFT模型——冻结所有原始参数，仅在q_proj和v_proj上注入可训练的LoRA旁路矩阵。
- **print_trainable_parameters**：打印可训练参数数量和占比。预期显示约0.3M可训练参数，占总参数0.065%左右。
- **为什么只注入q_proj和v_proj**：研究表明在注意力的Q和V投影上注入LoRA即可获得良好效果，注入更多模块（如k_proj、o_proj）效果提升有限但参数增加。这是性能与效率的权衡。

> **注意**：若报 `NameError: name 'LoraConfig' is not defined`，请先运行「3.2 安装依赖库」→依赖库导入单元格。

## 8. 开始训练

使用 HuggingFace `Trainer` 训练 5 个 epoch，batch_size=4，lr=2e-4。

In [ ]:
OUTPUT_DIR = './output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    warmup_steps=10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    report_to='none',
    fp16=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print('训练配置:')
print(f'  epochs = 5')
print(f'  batch_size = 4')
print(f'  learning_rate = 2e-4')
print(f'  fp16 = True')
print(f'  设备 = {device}')
print(f'\n开始训练...')

**代码说明**：配置训练参数——5个epoch（遍历数据集5次），batch_size=4（每步4个样本），learning_rate=2e-4（LoRA常用较大学习率），warmup_steps=10（前10步线性升温），max_grad_norm=1.0（梯度裁剪防爆炸），fp16=True（半精度训练加速）。使用HuggingFace Trainer封装训练循环。

> **注意**：若报 `NameError: name 'TrainingArguments' is not defined`，请先运行「3.2 安装依赖库」→依赖库导入单元格。

In [ ]:
start_time = time.time()
trainer.train()
elapsed = time.time() - start_time

print(f'\n训练完成! 耗时: {elapsed:.1f} 秒 ({elapsed/60:.1f} 分钟)')

**代码说明与预期结果**：

- **训练执行**：`trainer.train()`启动训练循环，每5步打印loss，训练过程中loss应逐步下降。
- **预期输出**：训练5个epoch（41条数据/batch_size 4 ≈ 10步/epoch × 5 = 约50步），在NPU上耗时约1~3分钟。训练完成打印总耗时。loss从初始值（约2~3）逐步下降到1左右。
- **为什么训练快**：LoRA仅训练0.065%的参数，反向传播计算量极小；fp16半精度加速；41条数据集很小。在NPU上整个训练几分钟内完成。

> **注意**：若报 `NameError: name 'time' is not defined`，请先运行「3.2 安装依赖库」→依赖库导入单元格。

## 9. 保存 LoRA 微调权重

LoRA 权重仅几 MB，相比完整模型（~924 MB）极大节省存储。

In [ ]:
save_path = os.path.join(OUTPUT_DIR, 'qwen_lora_finetuned')
model.save_pretrained(save_path)
print(f'权重已保存至: {save_path}')

print('文件清单:')
for f in os.listdir(save_path):
    size = os.path.getsize(os.path.join(save_path, f))
    print(f'  {f}: {size / 1024:.1f} KB')

**代码说明与预期结果**：`model.save_pretrained`保存LoRA权重到`./output/qwen_lora_finetuned/`。预期生成`adapter_model.safetensors`（LoRA权重，约几MB）和`adapter_config.json`（LoRA配置）。相比完整模型924MB，LoRA权重仅几MB，极大节省存储——这是LoRA的核心优势之一，每个微调任务只需存储几MB的适配器权重。

### 9.1 运行后 output 目录输出结果说明

训练完成后，`./output` 目录下共生成 3 个子目录：

```
output/
├── checkpoint-50/        # Trainer 在第50步自动保存的检查点
│   ├── adapter_config.json      # LoRA 配置（r、alpha、target_modules 等）
│   ├── adapter_model.safetensors# LoRA 旁路权重（低秩矩阵 B·A）
│   ├── optimizer.pt             # 优化器状态（Adam 动量等，用于断点续训）
│   ├── README.md                # PEFT 自动生成的适配器说明
│   ├── rng_state.pth            # 随机数发生器状态（恢复训练可复现）
│   ├── scaler.pt                # fp16 损失缩放器状态
│   ├── scheduler.pt             # 学习率调度器状态
│   ├── trainer_state.json       # Trainer 状态（loss 日志、全局步数等）
│   └── training_args.bin        # 训练参数快照
├── checkpoint-55/        # 训练结束（第55步）的最终检查点，文件同上9个
└── qwen_lora_finetuned/  # save_pretrained 保存的精简 LoRA 权重（用于推理部署）
    ├── adapter_config.json      # LoRA 配置
    ├── adapter_model.safetensors# LoRA 权重（约3MB）
    └── README.md                # PEFT 说明
```

**目录解读**：

- **checkpoint-50 / checkpoint-55**：HuggingFace `Trainer` 按 `save_steps=50` 在第50步和训练结束（第55步）自动保存的检查点。每个检查点含9个文件，除 LoRA 权重和配置外，还保存了优化器状态（`optimizer.pt`）、调度器状态（`scheduler.pt`）、随机数状态（`rng_state.pth`）、fp16 缩放器状态（`scaler.pt`）等，用于断点续训。`save_total_limit=2` 限制最多保留2个检查点，故 checkpoint-50 和 checkpoint-55 同时保留。
- **qwen_lora_finetuned**：由 `model.save_pretrained` 手动保存的精简 LoRA 权重，仅3个文件（`adapter_config.json` + `adapter_model.safetensors` + `README.md`），不含优化器等训练状态，体积约3MB。该目录供后续 `PeftModel.from_pretrained` 加载，实现"基座模型 + LoRA 适配器"的推理部署。
- **为什么 checkpoint 比 qwen_lora_finetuned 文件多**：checkpoint 面向**恢复训练**，需保存优化器/调度器等完整状态；qwen_lora_finetuned 面向**推理部署**，只需权重和配置即可。实际部署时只需 qwen_lora_finetuned，checkpoint 可在确认模型效果后删除以节省空间。

## 10. 微调后测试与效果对比

对相同问题进行测试，对比微调前后回答效果。

In [ ]:
model.eval()
print('=== 微调后模型测试 ===')
for q in TEST_QUESTIONS:
    print(f'  问: {q}')
    try:
        answer = ask_question(q, model, tokenizer, device)
        print(f'  答: {answer[:150]}')
    except Exception as e:
        print(f'  答: (出错: {e})')
    print()

# 泛化测试
extra_q = '什么是梯度下降？'
print(f'  [泛化测试] 问: {extra_q}')
try:
    answer = ask_question(extra_q, model, tokenizer, device)
    print(f'  答: {answer[:150]}')
except Exception as e:
    print(f'  答: (出错: {e})')

**代码说明与预期结果**：

- **微调后测试**：用与基线相同的3个问题测试微调后模型，对比回答质量。还额外测试"什么是梯度下降？"（训练集中有此问题）验证泛化效果。
- **预期输出**：微调后模型对昇腾相关问题的回答应更准确、更符合训练数据中的标准答案。例如"什么是昇腾910？"应回答包含"256TFLOPS"、"达芬奇架构"等训练数据中的关键信息。与微调前基线对比，回答质量和专业度应有明显提升。
- **为什么可能不完美**：41条数据、5个epoch的训练量有限，模型可能未完全记住所有答案。增加数据量或训练轮数可进一步提升效果，但需注意过拟合风险。

## 11. 加载微调权重进行推理

训练完成后，可使用以下代码加载微调后的模型进行推理：

In [ ]:
from peft import PeftModel

# 加载基础模型
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
).to(device)

# 加载 LoRA 权重
finetuned_model = PeftModel.from_pretrained(base_model, save_path)
finetuned_model = finetuned_model.to(device)
finetuned_model.eval()

# 推理测试
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': '什么是昇腾910？'},
]
input_ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors='pt',
    tokenize=True, return_dict=False,
).to(device)
with torch.no_grad():
    outputs = finetuned_model.generate(
        input_ids, max_new_tokens=128, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(outputs[0][input_ids.shape[-1]:], skip_special_tokens=True)
print(f'问: 什么是昇腾910？')
print(f'答: {response}')

**代码说明与预期结果**：

- **加载流程**：先加载基座模型`AutoModelForCausalLM.from_pretrained`，再用`PeftModel.from_pretrained`在其上加载LoRA适配器权重。推理时模型自动计算W = W₀ + (α/r)·B·A。
- **预期输出**：对"什么是昇腾910？"的回答应与微调后测试一致，验证保存的权重可正确恢复微调效果。
- **实际部署场景**：这模拟了实际部署流程——基座模型共享，不同任务的LoRA权重按需加载，实现"一个基座+多个任务适配器"的高效部署。

> **注意**：若报 `ModuleNotFoundError: No module named 'peft'`，请先运行「3.2 安装依赖库」单元格安装 peft。

## 12. 实验总结

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">指标</th>
<th style="text-align: left;">微调前</th>
<th style="text-align: left;">微调后</th>
</tr>
<tr>
<td style="text-align: left;">可训练参数</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">0.3M（0.065%）</td>
</tr>
<tr>
<td style="text-align: left;">总参数</td>
<td style="text-align: left;">462.0M</td>
<td style="text-align: left;">462.3M</td>
</tr>
<tr>
<td style="text-align: left;">昇腾 AI 问答准确率</td>
<td style="text-align: left;">较低</td>
<td style="text-align: left;">显著提升</td>
</tr>
<tr>
<td style="text-align: left;">LoRA 权重大小</td>
<td style="text-align: left;">—</td>
<td style="text-align: left;">几 MB</td>
</tr>
</table>

**表格解读**：本表格对比了微调前后的关键指标。可训练参数从0增至0.3M（仅占0.065%），这是LoRA的核心优势——用极少的参数实现有效微调。总参数从462.0M增至462.3M，增加的0.3M即为LoRA旁路矩阵B和A的参数。昇腾AI问答准确率从较低（基座模型对昇腾领域知识了解有限）到显著提升（微调后学会了训练数据中的专业知识）。LoRA权重大小仅几MB，相比完整模型924MB可忽略不计，使得多个微调版本的存储和切换成本极低。

训练数据集为内置的 41 条昇腾 AI 问答对，微调后模型能更准确、专业地回答华为昇腾 AI 生态相关问题。

**总结说明**：本实验完整演示了LoRA参数高效微调的流程——从数据准备、模型加载、LoRA配置、训练、权重保存到效果验证。关键结论是：仅训练0.065%的参数即可让模型学会特定领域知识，且LoRA权重仅几MB便于部署和切换。这使大模型微调从"需要高端GPU集群"降级为"单卡数分钟"，极大降低了定制化大模型的门槛。

---

## 13. 课后练习

**第1题**（单选题）LoRA 的核心思想是？

- A. 更新模型所有参数
- B. 冻结原始参数，仅训练旁路注入的低秩矩阵
- C. 剪枝去除冗余参数
- D. 知识蒸馏到小模型


In [ ]:
q1 = ''
print(f'第{1}题答案已记录：{q1}' if q1 else '⚠️ 请填入答案并运行本单元格')

**第2题**（单选题）本实验中 LoRA 的秩 r 和 alpha 分别是？

- A. r=4, alpha=8
- B. r=8, alpha=16
- C. r=16, alpha=32
- D. r=32, alpha=64


In [ ]:
q2 = ''
print(f'第{2}题答案已记录：{q2}' if q2 else '⚠️ 请填入答案并运行本单元格')

**第3题**（单选题）LoRA 注入旁路的目标模块是？

- A. q_proj, k_proj
- B. q_proj, v_proj
- C. k_proj, v_proj
- D. o_proj, q_proj


In [ ]:
q3 = ''
print(f'第{3}题答案已记录：{q3}' if q3 else '⚠️ 请填入答案并运行本单元格')

**第4题**（单选题）本实验可训练参数占总参数的比例约为？

- A. 0.065%
- B. 6.5%
- C. 65%
- D. 100%


In [ ]:
q4 = ''
print(f'第{4}题答案已记录：{q4}' if q4 else '⚠️ 请填入答案并运行本单元格')

**第5题**（单选题）本实验的训练数据集包含多少条问答对？

- A. 10 条
- B. 21 条
- C. 41 条
- D. 100 条


In [ ]:
q5 = ''
print(f'第{5}题答案已记录：{q5}' if q5 else '⚠️ 请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_03 import grade
grade(globals())

## 参考资料

- [Qwen 模型](https://huggingface.co/Qwen/Qwen1.5-0.5B-Chat)
- [PEFT 库](https://github.com/huggingface/peft)
- [昇腾 CANN 文档](https://www.hiascend.com/document)
- [LoRA 原始论文](https://arxiv.org/abs/2106.09685)